In [46]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [ ]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://opencode.ai/zen/v1/"
)

In [48]:
def llm(prompt):
    response = openai_client.chat.completions.create(
        model="deepseek-v4-flash-free",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

In [49]:
llm("Hey, what's up?")

"Hey! Not much on this end, just hanging out in the digital realm ready to help out. What's going on with you?"

In [50]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

That's exciting! I'd be happy to help you figure that out.

To give you the most accurate answer, could you let me know the **name of the course** you discovered? Many online courses (like those on Coursera, edX, or YouTube) are self-paced, which means you can join anytime. Other courses, especially live cohorts or university programs, have specific enrollment windows.

Once I know which course it is, I can help you look up whether enrollment is currently open or guide you to the right page to check.


In [51]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [52]:
answer = llm(prompt)
print(answer)

Based on the context, yes, you can join now. The course says: "I just discovered the course. Can I still join? Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions." It also states that you can start learning and submitting homework without registering.


## Dataset

In [1]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1349

## Search

In [54]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

search_results = search(question)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

## Building prompt

In [55]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [56]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [57]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [58]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [59]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

# The llm 

In [60]:
# we use chat.completions (as we have opencode go)

response = openai_client.chat.completions.create(
    model="deepseek-v4-flash-free",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

In [61]:
response


ChatCompletion(id='56cb3dde-d805-4d58-8e94-5b4a0aad4819', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Yes, based on the context provided, you can join now. The FAQ directly answers this question with: \n\n> **Yes**, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nTo give you the full picture based on the other FAQs:\n\n* **Registration:** You don\'t need to register first. You can just start learning and submitting homework while the form is open.\n* **Certificate:** You can only earn a certificate by participating in a "live" cohort and submitting your Capstone project while the peer-review window is open. The self-paced mode does not offer a certificate.\n* **Next Cohort:** If the project submission window for the current cohort has already closed, the course will be offered live again in **Summer 2025**.', refusal=None, role='assistant', annotations=None, au

In [62]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.total_tokens * output_price
)

cost

0.008019

In [ ]:
def llm(instructions, user_prompt, model="deepseek-v4-flash-free"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [ ]:
def rag(query, model="deepseek-v4-flash-free"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [68]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)


Yes, you can join now. According to the course FAQ, you can still join, but if you want to receive a certificate, you need to make sure you submit your project while submissions are still being accepted.


# rag helper

In [3]:
from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI
import os

documents = load_faq_data()
index = build_index(documents)

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://opencode.ai/zen/v1/"
)

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join the course. However, if you want to receive a certificate, you must submit your project while submissions are still being accepted.


# function call

In [5]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

response = openai_client.responses.create(
    model="mimo-v2.5-free",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"加入课程"}', call_id='call_81e043b3ae90421390271e19', name='search', type='function_call', id='call_81e043b3ae90421390271e19', namespace=None, status=None),
 ResponseFunctionToolCall(arguments='{"query":"课程注册"}', call_id='call_49dd51c1d42040f98d5a80c4', name='search', type='function_call', id='call_49dd51c1d42040f98d5a80c4', namespace=None, status=None),
 ResponseFunctionToolCall(arguments='{"query":"开始时间"}', call_id='call_10656ff926314bbab96e158b', name='search', type='function_call', id='call_10656ff926314bbab96e158b', namespace=None, status=None),
 ResponseFunctionToolCall(arguments='{"query":"还能加入"}', call_id='call_7037b35b75334849834b2fa3', name='search', type='function_call', id='call_7037b35b75334849834b2fa3', namespace=None, status=None)]

In [7]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [5]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [2]:
response = openai_client.responses.create(
    model="mimo-v2.5-free",
    input=messages,
    tools=[search_tool],
)

response.output_text

NameError: name 'openai_client' is not defined

## agentic loop

In [1]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [8]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="mimo-v2.5-free",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course enrollment late registration"}


In [11]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model="mimo-v2.5-free",
            input=messages,
            tools=[search_tool],
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [12]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally"}
iteration #2...
function_call: search {"query":"Ollama install run local"}
iteration #3...
function_call: search {"query":"Ollama run local server"}
iteration #4...
ASSISTANT:
Based on the course FAQ, here's how you can run **Ollama** locally:

## Installation
1. **Download Ollama** from [ollama.com/download](https://ollama.com/download) for your operating system:
   - **macOS**: Download and install the `.pkg` file
   - **Windows**: Download and install the `.msi` file
   - **Linux**: Run this terminal command:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

## Running Ollama Locally
Once installed, you can:

1. **Start a model** by opening a terminal and running:
   ```bash
   ollama run llama3
   ```
   This will:
   - Download the LLaMA 3 model (~4GB)
   - Start the model locally
   - Open a chat interface where you can type questions

2. **Test the local server** by running:
   ```bash
   cur

'Based on the course FAQ, here\'s how you can run **Ollama** locally:\n\n## Installation\n1. **Download Ollama** from [ollama.com/download](https://ollama.com/download) for your operating system:\n   - **macOS**: Download and install the `.pkg` file\n   - **Windows**: Download and install the `.msi` file\n   - **Linux**: Run this terminal command:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n## Running Ollama Locally\nOnce installed, you can:\n\n1. **Start a model** by opening a terminal and running:\n   ```bash\n   ollama run llama3\n   ```\n   This will:\n   - Download the LLaMA 3 model (~4GB)\n   - Start the model locally\n   - Open a chat interface where you can type questions\n\n2. **Test the local server** by running:\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should receive a response like: `{"models": [...]}`\n\n3. **Use the Python client** (optional):\n   ```bash\n   pip install ollama\n   ```\n   Then in Python:\n   ```python

In [13]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late"}
iteration #2...
function_call: search {"query":"accepting submissions deadline"}
iteration #3...
function_call: search {"query":"course schedule submission deadline"}
iteration #4...
function_call: search {"query":"LLM Zoomcamp 2024 submission deadline dates"}
iteration #5...
function_call: search {"query":"\"accepting submissions\" LLM Zoomcamp"}
iteration #6...
function_call: search {"query":"llm-zoomcamp start date"}
iteration #7...
function_call: search {"query":"LLM Zoomcamp schedule 2024"}
iteration #8...
function_call: search {"query":"\"while the form is open\" submissions"}
iteration #9...
function_call: search {"query":"Summer 2025 LLM Zoomcamp"}
iteration #10...
function_call: search {"query":"\"LLM Zoomcamp\" 2024 cohort"}
iteration #11...
ASSISTANT:
Yes, you can still join the LLM Zoomcamp course! 

According to the course FAQ, you're welcome to start learning and submitting homework even if you've just dis

'Yes, you can still join the LLM Zoomcamp course! \n\nAccording to the course FAQ, you\'re welcome to start learning and submitting homework even if you\'ve just discovered the course. However, if you want to receive a **certificate**, you must submit your project **while submissions are still being accepted** (the specific deadline varies by cohort). The next offering of the course is scheduled for **Summer 2025**.\n\n**Important notes:**\n- You don\'t need to register in advance—you can start immediately.\n- Certificates are only awarded to participants who complete the course with a "live" cohort (not in self-paced mode), because peer review of capstone projects is required.\n\nIf you\'d like more details about the current submission window, please check the course website or Slack channel for the latest updates.\n\nAre there other areas you\'d like to explore, such as registration details, course content, or certificate requirements?'

## frameworks

In [25]:
from toyaikit.llm import OpenAIChatCompletionsClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIChatCompletionsRunner, DisplayingRunnerCallback

In [23]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [ ]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [16]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [ ]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'No description provided.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [28]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIChatCompletionsRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIChatCompletionsClient(
        model="mimo-v2.5-free",
        client=OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"), base_url="https://opencode.ai/zen/v1/"
        ),
    ),
)

In [29]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


/workspaces/llm-zoomcamp-2026/01/rag/.venv/lib/python3.14/site-packages/toyaikit/chat/runners.py:542: UnknownModelWarning: No pricing data for model 'mimo-v2.5-free'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost = self.pricing_config.calculate_cost(


In [31]:
result.all_messages


[{'role': 'system',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'How do I run Olama locally?'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_e3c7a8a83dfb4810aae6c59d', function=Function(arguments='{"query": "Ollama locally"}', name='search'), type='function', index=0)], reasoning='The user is asking about running Ollama locally. Ollama is a tool for running large language models locally. Let me se

In [32]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


/workspaces/llm-zoomcamp-2026/01/rag/.venv/lib/python3.14/site-packages/toyaikit/chat/runners.py:542: UnknownModelWarning: No pricing data for model 'mimo-v2.5-free'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost = self.pricing_config.calculate_cost(
